## 1. Chuẩn hóa & Tổng hợp Metrics (`evaluation_results.csv`)

Cell code bên dưới thực hiện thu thập kết quả đánh giá của cả ba mô hình từ thư mục `results/metrics/`, chuẩn hóa về cùng một định dạng 9 chỉ số chung và xuất ra file tổng hợp cuối cùng:

- **Dữ liệu đầu vào (`results/metrics/`):**
  - `baseline_metrics.csv` (TF-IDF + Logistic Regression)
  - `frozen_encoder_metrics.csv` (DistilBERT Frozen Encoder)
  - `fine_tuning_metrics.csv` (DistilBERT Fine-Tuning)
- **Tệp kết quả đầu ra:** `results/metrics/evaluation_results.csv` *(tệp này nằm trong danh sách loại trừ của `.gitignore` nên sẽ tự động được sinh ra cục bộ trên máy khi chạy cell code bên dưới)*.

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

ROOT = Path("..").resolve() if Path.cwd().name == "notebooks" else Path(".").resolve()
METRICS_DIR = ROOT / "results" / "metrics"
FIGURES_DIR = ROOT / "results" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

LABEL_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

# 1. Chọn đúng 9 cột chuẩn chung từ các file metrics và đổi tên đẹp cho bảng báo cáo
COLS_TO_KEEP = [
    "model",
    "accuracy",
    "precision_macro",
    "recall_macro",
    "f1_macro",
    "train_time_s",
    "inference_time_s",
    "trainable_params",
    "model_size_mb",
]

RENAME_TO_DISPLAY = {
    "model": "Model",
    "accuracy": "Accuracy",
    "precision_macro": "Precision",
    "recall_macro": "Recall",
    "f1_macro": "Macro F1-score",
    "train_time_s": "Training Time (s)",
    "inference_time_s": "Inference Time (s)",
    "trainable_params": "Trainable Parameters",
    "model_size_mb": "Model Size (MB)",
}

metric_files = [
    METRICS_DIR / "baseline_metrics.csv",
    METRICS_DIR / "frozen_encoder_metrics.csv",
    METRICS_DIR / "fine_tuning_metrics.csv",  # Chờ Đức gửi thêm
]

frames = []
for f in metric_files:
    if f.exists():
        df = pd.read_csv(f)[COLS_TO_KEEP].rename(columns=RENAME_TO_DISPLAY)
        frames.append(df)
    else:
        print(f"Đang chờ file: {f.name}")

if frames:
    eval_df = pd.concat(frames, ignore_index=True)
    eval_df.to_csv(METRICS_DIR / "evaluation_results.csv", index=False)
    display(eval_df)

# 2. Vẽ và lưu Confusion Matrix từ các file predictions
pred_configs = [
    ("TF-IDF + Logistic Regression (Baseline)", METRICS_DIR / "baseline_test_predictions.csv", FIGURES_DIR / "confusion_matrix_baseline.png"),
    ("Frozen Encoder (DistilBERT)", METRICS_DIR / "frozen_encoder_predictions.csv", FIGURES_DIR / "confusion_matrix_frozen_encoder.png"),
    ("Fine-Tuning (DistilBERT)", METRICS_DIR / "fine_tuning_predictions.csv", FIGURES_DIR / "confusion_matrix_fine_tuning.png"),
]

for title, pred_path, fig_path in pred_configs:
    if pred_path.exists():
        df_pred = pd.read_csv(pred_path)
        cm = confusion_matrix(df_pred["true_label"], df_pred["pred_label"])
        plt.figure(figsize=(6.5, 5.5))
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
        plt.title(f"Confusion Matrix - {title}")
        plt.xlabel("Predicted Label")
        plt.ylabel("True Label")
        plt.tight_layout()
        plt.savefig(fig_path, dpi=300)
        plt.close()
        print(f"Đã lưu: {fig_path}")
    else:
        print(f"Đang chờ file predictions: {pred_path.name}")

Đang chờ file: fine_tuning_metrics.csv


,Model,Accuracy,Precision,Recall,Macro F1-score,Training Time (s),Inference Time (s),Trainable Parameters,Model Size (MB)
0,baseline,0.918158,0.918066,0.918158,0.917972,15.100099,0.309573,200004,3.409115
1,frozen_encoder,0.903421,0.903551,0.903421,0.903245,3289.519261,182.527057,3076,253.169937


Đã lưu: C:\PiCkis secrect place\Study documents\Deep Learning\Course Works\Course-Work-6\results\figures\confusion_matrix_baseline.png
Đã lưu: C:\PiCkis secrect place\Study documents\Deep Learning\Course Works\Course-Work-6\results\figures\confusion_matrix_frozen_encoder.png
Đang chờ file predictions: fine_tuning_predictions.csv
